# Videos → body points, on Kaggle or Colab

Turns exercise videos into MediaPipe Pose Lite points with the same settings as the project, **without downloading the videos to a laptop**. The output is one small `.npz` per video (points only, no pictures of people).

**On Kaggle (recommended):** create a notebook, *File → Import notebook* → this file. Then *Add Input* → search **wlu-rehabilitation-posture** → add. Settings: *Internet on*, *Accelerator none* (Pose Lite runs on the processor). Keep the notebook **private**. Choose *Save Version → Save & Run All*: it runs in the background, so you can close the laptop. Afterwards download `points.zip` from the *Output* tab.

**On Colab:** open this file and run all cells; the first cell downloads the dataset to Colab's disk through `kagglehub` (public datasets need no key).

Do not upload Fit3D here: its licence is for one person, on your own computer.

In [ ]:
%pip install -q "mediapipe>=0.10.14" opencv-python-headless

In [ ]:
import json, os, shutil, time, urllib.request
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

DATASET = "sulaimanmuhammed/wlu-rehabilitation-posture"   # <- any video dataset with one folder per class
LIMIT = None                                               # e.g. 5 for a quick try

ON_KAGGLE = Path("/kaggle/input").exists() and any(Path("/kaggle/input").iterdir())
if os.environ.get("VIDEO_ROOT"):                           # local test
    ROOT, WORK = Path(os.environ["VIDEO_ROOT"]), Path(os.environ["WORK_DIR"])
elif ON_KAGGLE:
    ROOT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
else:
    import kagglehub                                       # Colab: already installed there
    ROOT, WORK = Path(kagglehub.dataset_download(DATASET)), Path("/content")

OUT = WORK / "points"
OUT.mkdir(parents=True, exist_ok=True)
MODEL = WORK / "pose_landmarker_lite.task"
if not MODEL.exists():
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task", MODEL)
videos = sorted(p for p in ROOT.rglob("*") if p.suffix.lower() in (".mp4", ".mov", ".avi", ".mkv"))[:LIMIT]
print(len(videos), "videos; first:", videos[0] if videos else "none, did you add the dataset?")

In [ ]:
%%writefile extract_points.py
import sys
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np


def extract(job):
    """One video -> .npz with points (frames, 33, 5): x px, y px, z, visibility, presence. Missing person = NaN."""
    video, target, model = map(Path, job)
    if target.exists():
        return str(target), "already done"
    capture = cv2.VideoCapture(str(video))
    fps = capture.get(cv2.CAP_PROP_FPS) or 30.0
    width, height = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(model)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO, num_poses=1)
    rows, i = [], 0
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ok, bgr = capture.read()
            if not ok:
                break
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            found = landmarker.detect_for_video(image, int(i * 1000 / fps)).pose_landmarks
            rows.append([(m.x * width, m.y * height, (m.z or 0) * width, m.visibility or 0, m.presence or 0) for m in found[0]]
                        if found else [[np.nan] * 5] * 33)
            i += 1
    capture.release()
    points = np.asarray(rows, dtype=np.float32).reshape(-1, 33, 5)
    part = target.with_suffix(".part.npz")
    np.savez_compressed(part, points=points, fps=fps, width=width, height=height, source=video.name, folder=video.parent.name)
    part.rename(target)
    seen = float(np.isfinite(points[:, 0, 0]).mean()) if len(points) else 0.0
    return str(target), f"{len(points)} frames, person seen in {seen:.0%}"

In [ ]:
from extract_points import extract

def target_of(video):
    return OUT / ("__".join(video.relative_to(ROOT).with_suffix("").parts).replace(" ", "-") + ".npz")

jobs = [(str(v), str(target_of(v)), str(MODEL)) for v in videos]
start = time.time()
with ProcessPoolExecutor(max_workers=os.cpu_count()) as pool:
    for n, (name, note) in enumerate(pool.map(extract, jobs), 1):
        if n % 25 == 0 or n == len(jobs):
            print(f"{n}/{len(jobs)}  {Path(name).name}: {note}  ({time.time() - start:.0f} s)", flush=True)

In [ ]:
import collections
import numpy as np

counts = collections.Counter()
for f in OUT.glob("*.npz"):
    with np.load(f) as d:
        seen = np.isfinite(d["points"][:, 0, 0]).mean() if len(d["points"]) else 0
        counts[(str(d["folder"]), "ok" if seen > 0.8 else "person often missing")] += 1
for k, v in sorted(counts.items()):
    print(k, v)
archive = shutil.make_archive(str(WORK / "points"), "zip", OUT)
print("download this:", archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")

Put `points.zip` in `data/raw/<dataset>/` on the laptop. Training is light enough to run there; for a large network, attach `points.zip` to a Kaggle notebook with a graphics card instead.

**Before using any dataset:** check its licence, and whether files say *who* the person is. WLU does not number its people, so results on it cannot be split by person and must not be quoted as accuracy on new people.